# 03 — Perception review

Visually review deterministic media facts, shot boundaries, and representative frames.

Model-free: ffprobe + PySceneDetect + ffmpeg only. See `docs/perception.md`.

In [ ]:
# Bootstrap: make the local package importable when running from the repo.
import sys
from pathlib import Path

_src = Path.cwd()
if _src.name == 'notebooks':
    _src = _src.parent
if (_src / 'src' / 'tiktok_analytics_factory').is_dir():
    sys.path.insert(0, str(_src / 'src'))
import tiktok_analytics_factory  # noqa: E402
print(tiktok_analytics_factory.__file__)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tempfile

# Point at the ingested reference video (issue #2 output).
REFERENCE_VIDEO = Path("data/raw") / "<video_id>" / "video.mp4"

# Fallback for local development: generate a tiny synthetic video with known cuts.
if not REFERENCE_VIDEO.is_file():
    _ffmpeg = shutil.which("ffmpeg")
    if _ffmpeg is None:
        raise RuntimeError("no reference video found and ffmpeg unavailable for demo fixture")
    _tmp = Path(tempfile.mkdtemp())
    REFERENCE_VIDEO = _tmp / "demo_reference.mp4"
    subprocess.run(
        [_ffmpeg, "-v", "error"] + sum(
            [["-f", "lavfi", "-i", f"color=c={c}:s=64x64:d=1.2"] for c in ("red", "blue", "green", "yellow")],
            [],
        ) + ["-filter_complex", "concat=n=4:v=1:a=0",
             "-c:v", "libx264", "-pix_fmt", "yuv420p", "-y", str(REFERENCE_VIDEO)],
        check=True,
    )
    print(f"reference video not found; using synthetic demo fixture: {REFERENCE_VIDEO}")
else:
    print(f"using reference video: {REFERENCE_VIDEO}")

## 1. Run the perception pipeline

In [ ]:
from tiktok_analytics_factory.perception import run_perception

OUTPUT_DIR = Path("data/derived") / REFERENCE_VIDEO.stem / "perception" / "perception_v1"
manifest = run_perception(REFERENCE_VIDEO, OUTPUT_DIR)
manifest.to_dict().keys()

## 2. Media facts

In [ ]:
import json
print(json.dumps(manifest.media_facts.to_dict(), indent=2))

## 3. Shot boundaries

In [ ]:
for s in manifest.shots.shots:
    print(f"{s.shot_id}: {s.start_seconds:8.3f}s -> {s.end_seconds:8.3f}s "
          f"(frames {s.start_frame}-{s.end_frame})")

## 4. Manual hard-cut annotation

Edit this list after watching the video once: timestamps (seconds) of **hard visual cuts** only.

In [ ]:
# Hand-annotated hard visual cuts for the reference video.
MANUAL_CUTS = [1.2, 2.4, 3.6]  # <- replace with real annotations
MANUAL_CUTS

## 5. Boundary quality vs manual annotation (tolerance ±0.30 s)

In [ ]:
from tiktok_analytics_factory.perception import evaluate_boundaries
from tiktok_analytics_factory.perception.evaluation import CutAnnotation, shot_cut_timestamps

detected = shot_cut_timestamps([(s.start_seconds, s.end_seconds) for s in manifest.shots.shots])
ev = evaluate_boundaries(detected, [CutAnnotation(t) for t in MANUAL_CUTS],
                         tolerance_seconds=0.30)
ev.to_dict()

## 6. Visual review of representative frames

In [ ]:
from IPython.display import display, Image

for art in manifest.frames:
    print(f"{art.shot_id} @ {art.timestamp_seconds:.3f}s")
    display(Image(art.path, width=240))